In [ ]:
import joblib
import numpy as np

# ── Load model and encoder ────────────────────────────────────────────────────
model = joblib.load('models/lgbm_model.pkl')
le    = joblib.load('models/label_encoder.pkl')

print(f"Model loaded — {len(le.classes_)} diseases")

In [ ]:
#  ── Predict from a symptom vector ─────────────────────────────────────────────
# Replace with your actual feature column names
feature_cols = model.feature_name()  # LightGBM stores these automatically

def predict_disease(symptom_dict, top_k=3):
    """
    symptom_dict: {symptom_name: 0 or 1}
    Returns top_k diseases with probabilities.
    """
    # Build input vector
    X = np.zeros((1, len(feature_cols)), dtype='uint8')
    for i, col in enumerate(feature_cols):
        if symptom_dict.get(col, 0) == 1:
            X[0, i] = 1

    probs = model.predict_proba(X)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]

    results = [
        (le.inverse_transform([i])[0], round(probs[i] * 100, 2))
        for i in top_indices
    ]
    return results

# ── Example usage ─────────────────────────────────────────────────────────────
symptoms = {
    'fever': 1,
    'cough': 1,
    'shortness_of_breath': 1,
}

predictions = predict_disease(symptoms, top_k=3)
print("\nTop predictions:")
for disease, confidence in predictions:
    print(f"  {disease:<40} {confidence}%")